# Day 074 — Exercise 2: extract_frames

**What you'll build:** `extract_frames(source, step, max_frames, capture_fn) -> list[np.ndarray]` — extract frames as numpy arrays with step and count control.

**Why it matters:** A 2-hour movie at 30 FPS has 216,000 frames. `step` and `max_frames` make frame extraction feasible for any video length.

In [ ]:
import numpy as np
from pathlib import Path

def _make_test_frames(n=10, height=32, width=32):
    frames = []
    for i in range(n):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[:, :, 0] = int(255 * i / max(n - 1, 1))
        frames.append(frame)
    return frames


In [ ]:
_MOCK_META = {
    'fps': 30.0, 'frame_count': 10, 'width': 32, 'height': 32, 'duration_sec': 0.333,
}
_mock_info_fn    = lambda source: dict(_MOCK_META)
_mock_capture_fn = lambda source: _make_test_frames(10)
_mock_writer_fn  = lambda frames, path, fps: (Path(path).write_bytes(b'VIDEO' + bytes(len(frames))), Path(path))[1]
_mock_ffmpeg_fn  = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}


## Task

**Mock path:** `all_frames = capture_fn(source); stepped = all_frames[::step]; return stepped[:max_frames] if max_frames is not None else stepped`

**Real path:** `import cv2`, `VideoCapture` loop with `idx % step == 0` filter and `max_frames` early break.

## Your Implementation

In [ ]:
def extract_frames(source, step: int = 1, max_frames=None,
                   capture_fn=None) -> list:
    """Extract frames from a video as a list of numpy arrays (BGR, uint8).

    Args:
        source:     video file path
        step:       keep every nth frame (1=all, 2=every other, ...)
        max_frames: maximum frames to return (None=all)
        capture_fn: callable(source) -> list[np.ndarray] for testing
    """
    raise NotImplementedError


In [ ]:
def extract_frames(source, step=1, max_frames=None, capture_fn=None):
    if capture_fn is not None:
        all_frames = capture_fn(source)
        stepped    = all_frames[::step]
        return stepped[:max_frames] if max_frames is not None else stepped
    import cv2
    cap    = cv2.VideoCapture(str(source))
    frames = []
    idx    = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frames.append(frame)
            if max_frames is not None and len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # basic extraction — all frames
    frames = extract_frames('video.mp4', capture_fn=_mock_capture_fn)
    assert isinstance(frames, list) and len(frames) == 10
    score += 1; print("✅ extracts all 10 frames when step=1")

    # frame is numpy array with correct shape
    assert frames[0].shape == (32, 32, 3)
    assert frames[0].dtype.name == 'uint8'
    score += 1; print("✅ frames are (32, 32, 3) uint8 numpy arrays")

    # step parameter
    stepped = extract_frames('video.mp4', step=3, capture_fn=_mock_capture_fn)
    assert len(stepped) == 4, f"step=3 on 10 frames should give 4, got {len(stepped)}"
    score += 1; print("✅ step=3 selects correct frames (indices 0,3,6,9)")

    # max_frames cap
    capped = extract_frames('video.mp4', max_frames=3, capture_fn=_mock_capture_fn)
    assert len(capped) == 3, f"max_frames=3 should give 3, got {len(capped)}"
    score += 1; print("✅ max_frames=3 caps result at 3 frames")

    # step + max_frames combined
    combo = extract_frames('video.mp4', step=2, max_frames=3,
                           capture_fn=_mock_capture_fn)
    assert len(combo) <= 3
    score += 1; print("✅ step and max_frames work together")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_frames(source, step=1, max_frames=None, capture_fn=None):
    if capture_fn is not None:
        all_frames = capture_fn(source)
        stepped    = all_frames[::step]
        return stepped[:max_frames] if max_frames is not None else stepped
    import cv2
    cap    = cv2.VideoCapture(str(source))
    frames = []
    idx    = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frames.append(frame)
            if max_frames is not None and len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames
```

**Why `max_frames is not None`** rather than `if max_frames`? Because `max_frames=0` would be falsy but is a valid (if unusual) limit meaning return zero frames. The explicit `is not None` check is safer.

</details>